# CONTEST demo

This notebook explains one recommendation with CONTEST and with the REVISED+ baseline.

A unit is a test case cut at a milestone, for example after 40% of its events. The events before the cut are the history (or prefix) of the unit. For this history, the black box recommends the next activity and the resource to perform it. The explanation is a nearby alternative history, for which the black box would recommend a different activity and resource. Only the last events and the case attributes can change.

## Before you start (in a terminal)

Run these commands once, from the repository folder, with Python 3.11 or newer. If you use Anaconda, run `conda deactivate` first.

```bash
python3 -m venv venv
source venv/bin/activate          # on Windows: venv\Scripts\activate
python -m pip install -r requirements.txt notebook ipykernel
python -m ipykernel install --user --name contest --display-name "Python (CONTEST)"
```

In Jupyter, choose **Kernel → Change kernel → Python (CONTEST)**.

## 1. Setup

In [ ]:
import sys
print("Python", sys.version.split()[0], "at", sys.executable)
if "venv" not in sys.executable:
    print("WARNING: this is not the Python of the venv folder. See the README before installing.")
else:
    %pip install -q -r requirements.txt

## 2. Choose a unit

In [3]:
LOG = "bpi17_after"
CASE = "Application_1000610355"
MILESTONE = 0.4

import json, warnings
warnings.filterwarnings("ignore")
import pandas as pd

from contest.script import explain as explain_contest
from baseline.revised_plus.script import explain as explain_revised
from evaluation.report import find_unit, report
from evaluation.declare_plausibility import score

## 3. Run with CONTEST
This run could take a few minutes!

In [4]:
contest_run = explain_contest(LOG, case_ids=CASE, milestones=MILESTONE)
contest_run

Running CONTEST on case(s) Application_1000610355 of contest/case_studies/bpi17_after/test_data.csv (milestones 0.4) ...
Done in 1.1 min.
  Units in the results folder : 1
  Units with a counterfactual : 1
  Counterfactuals found       : 1
Results saved in results/contest/bpi17_after
  one JSON per unit in results/contest/bpi17_after/explanations/


{'output_dir': 'results/contest/bpi17_after',
 'explanations_dir': 'results/contest/bpi17_after/explanations',
 'units': 1,
 'units_with_cf': 1,
 'counterfactuals': 1,
 'skipped': 0,
 'minutes': 1.1}

## 4. Read the explanation

The report shows the explanation as in the worked example of the paper (Section 6). It has the original and counterfactual histories, the Contrastive Justification, the Downstream Consequences, and the Declare constraints the counterfactual satisfies or breaks.

When a unit has several counterfactuals, the report shows the one chosen as in the paper. Among those that improve the KPI by at least 5%, it is the one with the fewest changes. Use `show_all=True` to see all of them.

In [5]:
contest_unit = find_unit(contest_run["output_dir"], CASE, MILESTONE)
report(contest_unit,
       test_csv=f"contest/case_studies/{LOG}/test_data.csv",
       constraints=f"declare_constraints/{LOG}")


Case Application_1000610355 at milestone 40% (7 of 18 events)
After 7 events, the black box F recommends (O_Created, User_66) and forecasts 413.6 hours.
1 counterfactual(s) found. Shown: the KPI-improving one with the fewest changes.

  Step       Original                  Counterfactual           
  --------   -----------------------   -------------------------
  1 - 3      Identical                 Identical                
  4          W_Handle leads, User_5    A_Concept, User_1 *      
  5          A_Concept, User_5         A_Accepted, User_56 *    
  6          A_Accepted, User_51       O_Create Offer, User_56 *
  7          O_Create Offer, User_51   O_Created, User_38 *     
  8 (Rec.)   O_Created, User_66        A_Cancelled, User_1      
  --------   -----------------------   -------------------------
  * changed event

  Changed attributes:
    LoanGoal: Not speficied -> Other, see explanation
    RequestedAmount: 10000.00 -> 20025.48

  Contrastive Justification:
    The mode

## 6. Check Declare plausibility

`score` checks every counterfactual in a results folder against the Declare constraints mined from the training log, and returns one row per counterfactual. `declare_plausible_strict` is the plausibility reported in the paper, meaning no TDC and no LDC is broken. 

In [6]:
plausibility = score(train_csv=f"contest/case_studies/{LOG}/train_data.csv",
                     test_csv=f"contest/case_studies/{LOG}/test_data.csv",
                     explanations_dir=contest_run["explanations_dir"],
                     quiet=True)
plausibility[["case_id", "milestone", "cf_activity", "declare_plausible_strict",
              "tdc_checkable", "tdc_violations", "ldc_required", "ldc_violations"]]

,case_id,milestone,cf_activity,declare_plausible_strict,tdc_checkable,tdc_violations,ldc_required,ldc_violations
0,Application_1000610355,0.4,A_Cancelled,True,56,0,57,0


## 8. Run with REVISED+pres

REVISED+ is slower because it scores the Declare constraints at every search step.

In [7]:
revised_run = explain_revised(LOG, case_ids=CASE, milestones=MILESTONE)
revised_run

Running REVISED+ on case(s) Application_1000610355 of baseline/revised_plus/case_studies/bpi17_after/test_data.csv (milestones 0.4) ...
Done in 1.6 min.
  Units in the results folder : 1
  Units with a counterfactual : 1
  Counterfactuals found       : 3
Results saved in results/revised_plus/bpi17_after
  one JSON per unit in results/revised_plus/bpi17_after/explanations/


{'output_dir': 'results/revised_plus/bpi17_after',
 'explanations_dir': 'results/revised_plus/bpi17_after/explanations',
 'units': 1,
 'units_with_cf': 1,
 'counterfactuals': 3,
 'skipped': 0,
 'minutes': 1.6}

In [8]:
revised_unit = find_unit(revised_run["output_dir"], CASE, MILESTONE)
report(revised_unit,
       test_csv=f"baseline/revised_plus/case_studies/{LOG}/test_data.csv",
       constraints=f"declare_constraints/{LOG}")


Case Application_1000610355 at milestone 40% (7 of 18 events)
After 7 events, the black box F recommends (O_Created, User_66) and forecasts 413.6 hours.
3 counterfactual(s) found. Shown: the KPI-improving one with the fewest changes.

  Step       Original                  Counterfactual                     
  --------   -----------------------   -----------------------------------
  1 - 3      Identical                 Identical                          
  4          W_Handle leads, User_5    A_Concept, User_1 *                
  5          A_Concept, User_5         A_Accepted, User_56 *              
  6          A_Accepted, User_51       O_Create Offer, User_56 *          
  7          O_Create Offer, User_51   O_Sent (mail and online), User_56 *
  8 (Rec.)   O_Created, User_66        O_Accepted, User_100               
  --------   -----------------------   -----------------------------------
  * changed event

  Changed attributes:
    LoanGoal: Not speficied -> Existing loan tak